# Moduł odpowiadający za załadowanie i połączenie danych 

## Załadowanie danych pobranych z kaggle

In [1]:

import pandas as pd

Pobranie danych o częstości szkód

In [2]:
MTPL_freq = pd.read_csv('C:/Licencjat_DS/data/raw/freMTPLfreq.csv')
MTPL_freq.head()

,PolicyID,ClaimNb,Exposure,Power,CarAge,DriverAge,Brand,Gas,Region,Density
0,1,0,0.09,g,0,46,Japanese (except Nissan) or Korean,Diesel,Aquitaine,76
1,2,0,0.84,g,0,46,Japanese (except Nissan) or Korean,Diesel,Aquitaine,76
2,3,0,0.52,f,2,38,Japanese (except Nissan) or Korean,Regular,Nord-Pas-de-Calais,3003
3,4,0,0.45,f,2,38,Japanese (except Nissan) or Korean,Regular,Nord-Pas-de-Calais,3003
4,5,0,0.15,g,0,41,Japanese (except Nissan) or Korean,Diesel,Pays-de-la-Loire,60


In [3]:

print(MTPL_freq.shape)

(413169, 10)


In [4]:
MTPL_freq.columns

Index(['PolicyID', 'ClaimNb', 'Exposure', 'Power', 'CarAge', 'DriverAge',
       'Brand', 'Gas', 'Region', 'Density'],
      dtype='object')

Załadowanie danych o wysokości szkód

In [5]:
MTPL_sev = pd.read_csv('C:/Licencjat_DS/data/raw/freMTPLsev.csv')
MTPL_sev.head()

,PolicyID,ClaimAmount
0,63987,1172
1,310037,1905
2,314463,1150
3,318713,1220
4,309380,55077


In [6]:
MTPL_sev.shape

(16181, 2)

## Czyszczenie danych 

In [7]:
df_freq = MTPL_freq.copy()
df_sev = MTPL_sev.copy()

In [8]:
print("Frequency:", df_freq.shape)
print("Severity:", df_sev.shape)

print("Braki ClaimAmount:", df_sev["ClaimAmount"].isna().sum())
print("Braki IDpol:", df_sev["PolicyID"].isna().sum())

print("Unikalne polisy w frequency:", df_freq["PolicyID"].nunique())
print("Unikalne polisy w severity:", df_sev["PolicyID"].nunique())

Frequency: (413169, 10)
Severity: (16181, 2)
Braki ClaimAmount: 0
Braki IDpol: 0
Unikalne polisy w frequency: 413169
Unikalne polisy w severity: 15390


### Dane o częstości 

In [9]:

df_freq.describe()

,PolicyID,ClaimNb,Exposure,CarAge,DriverAge,Density
count,413169.000000,413169.000000,413169.000000,413169.000000,413169.000000,413169.000000
mean,206585.000000,0.039163,0.561088,7.532404,45.319876,1985.153913
std,119271.761023,0.204053,0.369477,5.763011,14.328120,4776.254543
min,1.000000,0.000000,0.002732,0.000000,18.000000,2.000000
25%,103293.000000,0.000000,0.200000,3.000000,34.000000,67.000000
50%,206585.000000,0.000000,0.540000,7.000000,44.000000,287.000000
75%,309877.000000,0.000000,1.000000,12.000000,54.000000,1410.000000
max,413169.000000,4.000000,1.990000,100.000000,99.000000,27000.000000


In [10]:
print(df_freq['PolicyID'].is_unique)

True


Zmiana wartości zmiennej ekspozycji z wartości powyżej 1 na 1

In [11]:
df_freq['Exposure'] = df_freq['Exposure'].clip(upper = 1.0)
print(df_freq['Exposure'].describe())

count    413169.000000
mean          0.560982
std           0.369304
min           0.002732
25%           0.200000
50%           0.540000
75%           1.000000
max           1.000000
Name: Exposure, dtype: float64


Przyciecie maksymalnej wartosci liczby szkod do jakiejs wartosci

In [12]:
max_claim_number = 4
df_freq['ClaimNb'] = df_freq['ClaimNb'].clip(upper = max_claim_number)

Sprawdzenie wartosci zmiennej claimnb

In [13]:
print(df_freq["ClaimNb"].value_counts().sort_index())

print("Odsetek polis ze szkodą:",
      (df_freq["ClaimNb"] > 0).mean())

ClaimNb
0    397779
1     14633
2       726
3        28
4         3
Name: count, dtype: int64
Odsetek polis ze szkodą: 0.0372486803220958


In [14]:
df_freq.to_csv('C:/Licencjat_DS/data/processed/freq.csv')

### Dane o wysokosciach szkody

In [15]:

df_sev.describe()

,PolicyID,ClaimAmount
count,16181.000000,1.618100e+04
mean,185015.202151,2.129972e+03
std,117946.980398,2.106364e+04
min,33.000000,2.000000e+00
25%,83922.000000,6.980000e+02
50%,169907.000000,1.156000e+03
75%,281599.000000,1.243000e+03
max,412944.000000,2.036833e+06


Brak ewidentnych obserwacji do odrzucenia

Kolejne zadanie - agregacja danych o wysokosci polisy ze wzgledu na id polisy

In [16]:
severity_agg = (
    df_sev
    .groupby("PolicyID", as_index=False)
    .agg(
        TotalClaimAmount=("ClaimAmount", "sum"),
        ClaimCount=("ClaimAmount", "count")
    )
)


In [17]:
severity_agg.head()

,PolicyID,TotalClaimAmount,ClaimCount
0,33,302,1
1,41,2001,1
2,92,1449,1
3,96,10870,2
4,142,1390,1


In [18]:
severity_agg.to_csv('C:/Licencjat_DS/data/processed/severity.csv')

## Łączenie danych w jeden zbiór

In [19]:
print(df_freq['PolicyID'].dtype)
print(severity_agg['PolicyID'].dtype)

int64
int64


Kontrola zbiorów danych

In [20]:
print("=== FREQUENCY ===")
print("Shape:", df_freq.shape)
print("Unikalne polisy:", df_freq["PolicyID"].nunique())
print("Brakujące ID:", df_freq["PolicyID"].isna().sum())
print("Duplikaty ID:", df_freq["PolicyID"].duplicated().sum())

print("\n=== SEVERITY ===")
print("Shape:", df_sev.shape)
print("Unikalne polisy:", df_sev["PolicyID"].nunique())
print("Brakujące ID:", df_sev["PolicyID"].isna().sum())
print("Brakujące kwoty:", df_sev["ClaimAmount"].isna().sum())

=== FREQUENCY ===
Shape: (413169, 10)
Unikalne polisy: 413169
Brakujące ID: 0
Duplikaty ID: 0

=== SEVERITY ===
Shape: (16181, 2)
Unikalne polisy: 15390
Brakujące ID: 0
Brakujące kwoty: 0


Sprawdzenie czy PolicyID pasują do siebie

In [21]:
freq_ids = set(df_freq["PolicyID"].dropna())
sev_ids = set(df_sev["PolicyID"].dropna())

print("Polisy w frequency:", len(freq_ids))
print("Polisy w severity:", len(sev_ids))
print("Wspólne polisy:", len(freq_ids & sev_ids))
print("Tylko frequency:", len(freq_ids - sev_ids))
print("Tylko severity:", len(sev_ids - freq_ids))

Polisy w frequency: 413169
Polisy w severity: 15390
Wspólne polisy: 15390
Tylko frequency: 397779
Tylko severity: 0


Czy polisy z czestosci wystepuja w wielkosci szkod

In [22]:
claims = df_freq.loc[df_freq["ClaimNb"] > 0]

print("Polisy z ClaimNb > 0:", len(claims))
print(
    "Z dopasowaniem w severity:",
    claims["PolicyID"].isin(sev_ids).sum()
)
print(
    "Bez dopasowania:",
    (~claims["PolicyID"].isin(sev_ids)).sum()
)

Polisy z ClaimNb > 0: 15390
Z dopasowaniem w severity: 15390
Bez dopasowania: 0


Czy liczba szkod zgadza sie z liczba rekordow w severity

In [23]:
severity_counts = df_sev.groupby("PolicyID").size()

check = (
    df_freq[["PolicyID", "ClaimNb"]]
    .merge(
        severity_counts.rename("SeverityCount"),
        on="PolicyID",
        how="left",
        validate="one_to_one"
    )
)

check["SeverityCount"] = check["SeverityCount"].fillna(0).astype(int)
check["Difference"] = check["ClaimNb"] - check["SeverityCount"]

print("Suma ClaimNb:", df_freq["ClaimNb"].sum())
print("Liczba rekordów severity:", len(df_sev))
print("Polisy z różną liczbą szkód:", (check["Difference"] != 0).sum())

print(
    check.loc[check["Difference"] != 0]
    .head(10)
)

Suma ClaimNb: 16181
Liczba rekordów severity: 16181
Polisy z różną liczbą szkód: 0
Empty DataFrame
Columns: [PolicyID, ClaimNb, SeverityCount, Difference]
Index: []


In [24]:
df_clean = pd.merge(df_freq, severity_agg, how = 'left', on = 'PolicyID')

In [25]:
df_clean.head()

,PolicyID,ClaimNb,Exposure,Power,CarAge,DriverAge,Brand,Gas,Region,Density,TotalClaimAmount,ClaimCount
0,1,0,0.09,g,0,46,Japanese (except Nissan) or Korean,Diesel,Aquitaine,76,NaN,NaN
1,2,0,0.84,g,0,46,Japanese (except Nissan) or Korean,Diesel,Aquitaine,76,NaN,NaN
2,3,0,0.52,f,2,38,Japanese (except Nissan) or Korean,Regular,Nord-Pas-de-Calais,3003,NaN,NaN
3,4,0,0.45,f,2,38,Japanese (except Nissan) or Korean,Regular,Nord-Pas-de-Calais,3003,NaN,NaN
4,5,0,0.15,g,0,41,Japanese (except Nissan) or Korean,Diesel,Pays-de-la-Loire,60,NaN,NaN


Czy istnieja obserwacje, gdzie ClaimNb > 0, ale nie mamy informacji o wielkosci szkody

In [26]:
print(
    df_clean.loc[
        df_clean["ClaimNb"] > 0,
        ["TotalClaimAmount", "ClaimCount"]
    ].isna().sum()
)

TotalClaimAmount    0
ClaimCount          0
dtype: int64


Zamieniamy NA na 0 - brak szkody to brak kwoty i brak lacznej liczby szkod

In [29]:
df_clean['TotalClaimAmount'].fillna(0, inplace = True)
df_clean['ClaimCount'].fillna(0, inplace = True)

C:\Users\kadzp\AppData\Local\Temp\ipykernel_24156\2316981397.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['TotalClaimAmount'].fillna(0, inplace = True)
C:\Users\kadzp\AppData\Local\Temp\ipykernel_24156\2316981397.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

F

In [30]:
df_clean.isna().sum()

PolicyID            0
ClaimNb             0
Exposure            0
Power               0
CarAge              0
DriverAge           0
Brand               0
Gas                 0
Region              0
Density             0
TotalClaimAmount    0
ClaimCount          0
dtype: int64

In [31]:
df_clean.to_csv('C:/Licencjat_DS/data/processed/df_merged.csv')